In [ ]:
!pip install roboflow timm scikit-learn matplotlib seaborn grad-cam -q
print("Installed ✓")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")

In [ ]:
import os, shutil, random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from PIL import Image
from roboflow import Roboflow
print("Imports done ✓")

In [ ]:
rf = Roboflow(api_key="YOUR_API_KEY_HERE")

project = rf.workspace("sina-us3z2").project("dental-caries-classification-otjgb")
project.version(1).download("folder")

project = rf.workspace("sina-us3z2").project("without-cavity-6o1ul")
project.version(1).download("folder")
print("Datasets downloaded ✓")

In [ ]:
base = '/content/data'
os.makedirs(f'{base}/caries', exist_ok=True)
os.makedirs(f'{base}/normal', exist_ok=True)

d1 = '/content/Dental-Caries-Classification-1'
for split in ['train', 'valid', 'test']:
    for cls in ['caries', 'deep caries']:
        src = os.path.join(d1, split, cls)
        if os.path.exists(src):
            for f in os.listdir(src):
                shutil.copy(os.path.join(src, f), f'{base}/caries/{f}')
    src = os.path.join(d1, split, 'null')
    if os.path.exists(src):
        for f in os.listdir(src):
            shutil.copy(os.path.join(src, f), f'{base}/normal/{f}')

d2 = '/content/Without-Cavity-1/train/without_cavity'
if os.path.exists(d2):
    for f in os.listdir(d2):
        shutil.copy(os.path.join(d2, f), f'{base}/normal/{f}')

print(f"Caries: {len(os.listdir(f'{base}/caries'))}")
print(f"Normal: {len(os.listdir(f'{base}/normal'))}")

In [ ]:
random.seed(42)
base_out = '/content/dataset'

for split in ['train', 'val', 'test']:
    for cls in ['caries', 'normal']:
        os.makedirs(f'{base_out}/{split}/{cls}', exist_ok=True)

for cls in ['caries', 'normal']:
    images = [f for f in os.listdir(f'{base}/{cls}')
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)
    n = len(images)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    for fname in images[:n_train]:
        shutil.copy(f'{base}/{cls}/{fname}', f'{base_out}/train/{cls}/{fname}')
    for fname in images[n_train:n_train+n_val]:
        shutil.copy(f'{base}/{cls}/{fname}', f'{base_out}/val/{cls}/{fname}')
    for fname in images[n_train+n_val:]:
        shutil.copy(f'{base}/{cls}/{fname}', f'{base_out}/test/{cls}/{fname}')
    print(f"{cls}: {n_train} train | {n_val} val | {n-n_train-n_val} test")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('/content/dataset/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder('/content/dataset/val',   transform=val_transforms)
test_dataset  = datasets.ImageFolder('/content/dataset/test',  transform=val_transforms)

class_counts = np.array([len(os.listdir(f'/content/dataset/train/{c}'))
                          for c in train_dataset.classes])
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for _, label in train_dataset.samples]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_val_acc = 0.0
for epoch in range(20):
    model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = train_correct / len(train_dataset)
    val_acc   = val_correct / len(val_dataset)
    print(f"Epoch {epoch+1:02d}/20 | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pth')
        print(f"  ✓ Best model saved (val_acc={val_acc:.4f})")

    scheduler.step()

print(f"\nTraining complete. Best Val Accuracy: {best_val_acc:.4f}")

In [ ]:
# Load best model
model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval()

# Test set evaluation
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_dataset.classes,
            yticklabels=test_dataset.classes)
plt.title('Confusion Matrix — EfficientNet-B0 Caries Classifier')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()
print("Confusion matrix done ✓")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from PIL import Image

caries_dir = '/content/dataset/test/caries'
sample_images = os.listdir(caries_dir)[:4]
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

cam = GradCAM(model=model, target_layers=[model.conv_head])
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, fname in enumerate(sample_images):
    img = Image.open(f'{caries_dir}/{fname}').convert('RGB')
    img_resized = img.resize((224, 224))
    img_array = np.array(img_resized) / 255.0
    input_tensor = preprocess(img).unsqueeze(0).to(device)
    grayscale_cam = cam(input_tensor=input_tensor,
                        targets=[ClassifierOutputTarget(0)])[0]
    visualization = show_cam_on_image(img_array.astype(np.float32),
                                      grayscale_cam, use_rgb=True)
    axes[0][i].imshow(img_resized, cmap='gray')
    axes[0][i].set_title('Original')
    axes[0][i].axis('off')
    axes[1][i].imshow(visualization)
    axes[1][i].set_title('Grad-CAM')
    axes[1][i].axis('off')

plt.suptitle('Grad-CAM: Where the Model Detects Caries', fontsize=14)
plt.tight_layout()
plt.savefig('/content/gradcam.png', dpi=150)
plt.show()
print("Grad-CAM done ✓")

# Save everything to Drive
save_dir = '/content/drive/MyDrive/dental_ai_project'
os.makedirs(save_dir, exist_ok=True)
shutil.copy('/content/best_model.pth',       f'{save_dir}/best_model.pth')
shutil.copy('/content/confusion_matrix.png', f'{save_dir}/confusion_matrix.png')
shutil.copy('/content/gradcam.png',          f'{save_dir}/gradcam.png')
print("All files saved to Drive ✓")